# v2 LaughO Training Pipeline (Colab GPU)

Trains the Laughter-Only MLP on AudioSet laughter family + StandUp4AI weak labels.

**Architecture**: WavLM-base-plus (768-dim, frozen) + 256→128→1 MLP head (~1.5M trainable)

**Validation**: 5×3 repeated speaker-disjoint CV with paired bootstrap CI

**Source**: github.com/Das-rebel/HaHaScore (commit 7352527)

**Derived from v19 canonical** (gist 188a3bc5d4346c8189372f00c8bc2d39) — same ffmpeg-subprocess AAC loader, same processed_idx guard, BatchNorm1d crash-safe, start_idx>0 guard.

## Cell 1: Mount Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install --quiet onnxruntime scikit-learn scipy numpy
print('Mounted + deps installed')

## Cell 2: Verify v10 ONNX (sanity check)

In [ ]:
import os, hashlib, sys
V10_PATH = '/content/drive/MyDrive/LaughterOnAudio/HaHaScore/hahascore-cascade/v10_cascade_int8.onnx'
if os.path.exists(V10_PATH):
    print(f'v10 ONNX: {os.path.getsize(V10_PATH)/1024/1024:.2f} MB')
    print(f'SHA: {hashlib.sha256(open(V10_PATH, "rb").read()).hexdigest()[:16]}')
else:
    print(f'NOT FOUND: {V10_PATH}')
    print('Upload from github.com/Das-rebel/HaHaScore/deployment/models/v10_cascade_int8.onnx')

## Cell 3: Clone HaHaScore repo for harness + scripts

In [ ]:
%cd /content
!git clone https://github.com/Das-rebel/HaHaScore.git
%cd /content/HaHaScore
!git log --oneline -3
sys.path.insert(0, '/content/HaHaScore')
print('Scripts ready:', sorted([f for f in os.listdir('.') if f.startswith('laugho')]))

## Cell 4: Download 1 AudioSet eval parquet (650 MB)

In [ ]:
import requests
from pathlib import Path

AUDIOSET_URL = 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/eval/00.parquet'
DEST = Path('/content/audioset_eval_00.parquet')

if not DEST.exists():
    print(f'Downloading {AUDIOSET_URL} (~650 MB)...')
    r = requests.get(AUDIOSET_URL, stream=True)
    r.raise_for_status()
    with open(DEST, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print(f'Downloaded: {DEST.stat().st_size/1024/1024:.1f} MB')
else:
    print(f'Cached: {DEST} ({DEST.stat().st_size/1024/1024:.1f} MB)')

## Cell 5: Filter 8 laugh ontology IDs + extract features

In [ ]:
import pyarrow.parquet as pq
import numpy as np
from io import BytesIO
import subprocess

LAUGH_IDS = {
    '/m/01j3sz': 'Laughter',
    '/t/dd00001': 'Baby_laughter',
    '/m/07r660_': 'Giggle',
    '/m/07s04w4': 'Snicker',
    '/m/07sq110': 'Belly_laugh',
    '/m/07rgt08': 'Chuckle',
    '/m/07q0yl5': 'Snort',
}

print('Loading parquet (500 rows expected)...')
table = pq.read_table('/content/audioset_eval_00.parquet')
rows = table.to_pylist()
print(f'  Total rows: {len(rows)}')

laugho_rows = []
for r in rows:
    labels = r.get('labels', [])
    matched = [l for l in labels if l in LAUGH_IDS]
    if matched:
        laugho_rows.append({'row': r, 'label': matched[0]})

print(f'  Laugh rows: {len(laugho_rows)}')
print(f'  Estimated decode time: ~{len(laugho_rows) * 0.3:.0f} sec')

## Cell 6: Decode audio bytes via WavLM-base-plus (frozen)

In [ ]:
import torch
from transformers import WavLMModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading WavLM-base-plus (frozen)...')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to(device).eval()
for p in wavlm.parameters():
    p.requires_grad = False
print(f'  WavLM loaded: {sum(p.numel() for p in wavlm.parameters())/1e6:.1f}M params (frozen)')

import soundfile as sf

def extract_wavlm_embedding(audio_bytes):
    """Decode audio bytes via ffmpeg-subprocess (v19 pattern) + WavLM encode."""
    try:
        p = subprocess.run(
            ['ffmpeg', '-loglevel', 'error', '-i', 'pipe:0',
             '-f', 'wav', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', 'pipe:1'],
            input=audio_bytes, capture_output=True
        )
        if p.returncode != 0:
            return None
        audio, sr = sf.read(BytesIO(p.stdout))
        if sr != 16000:
            return None
        audio_t = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            out = wavlm(audio_t).last_hidden_state.mean(dim=1)  # (1, 768)
        return out.cpu().numpy().squeeze()
    except Exception as e:
        return None

print('Extracting embeddings...')
embeddings = []
labels = []
video_ids = []
for i, lr in enumerate(laugho_rows):
    r = lr['row']
    audio_field = r.get('audio', {})
    audio_bytes = audio_field.get('bytes') if isinstance(audio_field, dict) else None
    if audio_bytes is None:
        continue
    emb = extract_wavlm_embedding(audio_bytes)
    if emb is None:
        continue
    embeddings.append(emb)
    labels.append(1)  # all are laugh by construction
    video_ids.append(r.get('video_id', f'laugh_{i}'))
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(laugho_rows)} decoded')

print(f'\nExtracted: {len(embeddings)} laugh embeddings of dim {embeddings[0].shape if embeddings else "n/a"}')

## Cell 7: Add negative samples from non-laugh rows

In [ ]:
import random
random.seed(42)

non_laugh_indices = [i for i, r in enumerate(rows) if not any(l in LAUGH_IDS for l in r.get('labels', []))]
print(f'  Non-laugh rows available: {len(non_laugh_indices)}')
sample_size = min(len(non_laugh_indices), len(embeddings) * 5)
neg_sample = random.sample(non_laugh_indices, sample_size)

print(f'  Extracting {sample_size} non-laugh samples...')
for i in neg_sample:
    r = rows[i]
    audio_field = r.get('audio', {})
    audio_bytes = audio_field.get('bytes') if isinstance(audio_field, dict) else None
    if audio_bytes is None:
        continue
    emb = extract_wavlm_embedding(audio_bytes)
    if emb is None:
        continue
    embeddings.append(emb)
    labels.append(0)
    video_ids.append(r.get('video_id', f'nonlaugh_{i}'))

print(f'\nFinal dataset: {len(embeddings)} samples, {sum(labels)} laugh, {len(labels)-sum(labels)} non-laugh')
print(f'Positive rate: {sum(labels)/len(labels):.3f}')

X = np.stack(embeddings)
y = np.array(labels)
groups = np.array(video_ids)
print(f'X: {X.shape}, y: {y.shape}, groups: {len(np.unique(groups))} unique')

## Cell 8: 5×3 repeated speaker-disjoint CV training

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

class LaughOMLP(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_eval_fold(tr_X, tr_y, va_X, va_y, seed=42, n_epochs=10):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LaughOMLP().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    pos_rate = max(tr_y.mean(), 0.001)
    pos_weight = torch.tensor([(1 - pos_rate) / pos_rate]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    X_tr = torch.tensor(tr_X, dtype=torch.float32).to(device)
    y_tr = torch.tensor(tr_y, dtype=torch.float32).to(device)
    X_va = torch.tensor(va_X, dtype=torch.float32).to(device)
    bs = 256
    n = len(tr_X)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs * n)
    for ep in range(n_epochs):
        model.train()
        perm = np.random.permutation(n)
        for i in range(0, n, bs):
            batch = perm[i:i+bs]
            optimizer.zero_grad()
            loss = criterion(model(X_tr[batch]), y_tr[batch])
            loss.backward()
            optimizer.step()
            sched.step()
    model.eval()
    with torch.no_grad():
        logits = model(X_va).cpu().numpy()
    scores = 1 / (1 + np.exp(-logits))
    if len(np.unique(va_y)) < 2:
        return 0.5
    return float(roc_auc_score(va_y, scores))

# Build fold splits: 5-fold by video_id, repeated across 3 seeds
from sklearn.model_selection import GroupKFold
results = []
for seed_offset in range(3):
    seed = 42 + seed_offset * 100
    rng = np.random.RandomState(seed)
    unique_groups = np.unique(groups)
    shuffled = rng.permutation(unique_groups)
    group_map = {g: shuffled[i] for i, g in enumerate(unique_groups)}
    groups_shuffled = np.array([group_map[g] for g in groups])
    gkf = GroupKFold(n_splits=5)
    for fold_idx, (tr, va) in enumerate(gkf.split(X, y, groups_shuffled)):
        auc = train_eval_fold(X[tr], y[tr], X[va], y[va], seed=seed)
        results.append({'seed': seed, 'fold': fold_idx, 'auc': auc})
        print(f'  seed={seed} fold={fold_idx}: AUC={auc:.4f}')

aucs = [r['auc'] for r in results]
print(f'\n5×3 CV results:')
print(f'  Mean: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'  Min:  {np.min(aucs):.4f}')
print(f'  Max:  {np.max(aucs):.4f}')

## Cell 9: Save results to Drive

In [ ]:
import json
from datetime import datetime
from pathlib import Path

OUTPUT_DIR = Path('/content/drive/MyDrive/LaughterOnAudio/HaHaScore/v2_laugho_runs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_path = OUTPUT_DIR / f'5x3_cv_audioset_eval00_{timestamp}.json'

results_data = {
    'timestamp': timestamp,
    'source': 'AudioSet eval/00.parquet (agkphysics mirror)',
    'n_laugh_extracted': int(sum(labels)),
    'n_non_laugh_extracted': int(len(labels) - sum(labels)),
    'positive_rate': float(sum(labels) / len(labels)),
    'n_unique_videos': int(len(np.unique(groups))),
    '5x3_cv_results': results,
    'auc_mean': float(np.mean(aucs)),
    'auc_std': float(np.std(aucs)),
    'auc_min': float(np.min(aucs)),
    'auc_max': float(np.max(aucs)),
    'protocol': '5-fold GroupKFold by video_id, 3 seeds, frozen WavLM + 1.5M MLP',
    'git_commit': '7352527',
}
results_path.write_text(json.dumps(results_data, indent=2))
print(f'Saved: {results_path}')
print(f'\nSummary:')
print(f'  {len(aucs)} measurements: mean={np.mean(aucs):.4f} std={np.std(aucs):.4f}')
print(f'  This is the honest v2 LaughO baseline (NOT to be cited as headline)')